# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MRazaRashid/FlyRank_Week1/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import os, sys, subprocess, pandas as pd
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found.")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Working dir: /content/flyrank-ml-internship-starter
Starter data found.


In [18]:
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,staleness_bucket
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,<90d
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,<90d
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,<90d
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,<90d
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,<90d


In [26]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, float("inf")],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
staleness_check = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
).reset_index()

print(staleness_check)

  staleness_bucket      n  decline_rate
0             <90d  20655      0.512031
1          90-180d   9171      0.611057
2         180-365d    169      0.467456
3            365d+      5      0.600000


Verdict: MIXED. Among buckets with sufficient sample size (<90d: n=20,655, decline_rate=51.2%; 90-180d: n=9,171, decline_rate=61.1%), staleness shows a real, if modest, positive relationship with decline. The more time since update does correlate with more decline. However, the 180-365d and 365d+ buckets have far too few rows (n=169 and n=5) to draw any conclusion, and their numbers contradict the trend, likely due to small-sample noise rather than a genuine reversal. This dataset is heavily skewed toward recently-updated content, which limits how strongly I can claim staleness as a signal beyond the first ~180 days.

# **Signal Check 2 CTR VS POSITION**

In [31]:
df["position_tier"] = pd.cut(
    df["avg_position"],
    bins=[-1,0,3,10,20,float('inf')],
    labels=["No_data","1-3","4-10","11-20","20+"]
)
ctr_check=df[df['avg_position']>0].groupby('position_tier',observed=True).agg(
    n=('content_id','count'),
    avg_ctr=('ctr','mean')).reset_index()
ctr_check

,position_tier,n,avg_ctr
0,1-3,1141,2.714303
1,4-10,11842,0.651045
2,11-20,7273,0.323443
3,20+,8539,0.211333


Verdict: CONFIRMED. CTR declines sharply and consistently as position worsens: pages ranking 1-3 average 2.71% CTR, dropping to 0.65% at positions 4-10, 0.32% at 11-20, and just 0.21% at 20+. Every bucket has a large, trustworthy sample size (n ≥ 1,141), and the pattern is clean and monotonic. No reversals like the staleness check showed. This confirms the CTR-vs-position relationship that FlyRank's real CTR-fix logic relies on: a page's expected CTR should be judged relative to its position tier, not in isolation, since even a "good" 0.5% CTR is actually excellent at position 20+ but poor at position 1-3.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [32]:
import os
import numpy as np

def assign_reason_and_action(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page", "refresh"
    elif row["trend_direction"] == "down" and row["impressions_90d"] >= 100:
        return "declining_with_demand", "review"
    elif row["avg_position"] > 0 and row["avg_position"] <= 20 and row["ctr"] < 0.5 and row["impressions_90d"] >= 500:
        return "low_ctr_visible_page", "improve_metadata"
    else:
        return "no_flag", "monitor"

df[["reason_code", "action"]] = df.apply(
    lambda row: pd.Series(assign_reason_and_action(row)), axis=1
)

# Simple transparent score: weighted combination of normalized signals
df["staleness_score"] = (df["days_since_last_update"] / df["days_since_last_update"].max()).clip(0, 1)
df["demand_score"] = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"]).max()
df["decline_score"] = (df["trend_direction"] == "down").astype(float)

df["baseline_action_score"] = (
    0.4 * df["staleness_score"] +
    0.35 * df["demand_score"] +
    0.25 * df["decline_score"]
) * 100

ranked = df.sort_values("baseline_action_score", ascending=False)[
    ["content_id", "client_id", "baseline_action_score", "reason_code", "action",
     "impressions_90d", "trend_direction", "days_since_last_update", "avg_position", "ctr"]
]

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked))
ranked.head(20)

Rows written: 30000


,content_id,client_id,baseline_action_score,reason_code,action,impressions_90d,trend_direction,days_since_last_update,avg_position,ctr
26840,content_7f116ae1f6f5,client_9400f1b21c,75.531961,stale_visible_page,refresh,954,down,301,9.0,0.42
16751,content_cf56e2e2e282,client_7f2253d7e2,75.144877,stale_visible_page,refresh,61678,down,194,19.7,0.15
7452,content_72496874f806,client_4ec9599fc2,75.133017,stale_visible_page,refresh,821,down,301,5.8,0.24
16514,content_7368877ea310,client_7f2253d7e2,75.047992,stale_visible_page,refresh,59472,down,194,24.8,0.13
26242,content_55a5b1c46474,client_4ec9599fc2,74.532677,no_flag,monitor,35,down,373,7.5,0.00
7021,content_1bfaa38ff26c,client_7f2253d7e2,72.817703,stale_visible_page,refresh,25715,down,194,22.2,0.23
21984,content_02b0d6e30129,client_19581e27de,72.334988,declining_with_demand,review,176,down,313,6.9,0.00
23741,content_df1fa766cac2,client_9400f1b21c,71.786336,declining_with_demand,review,206,down,304,4.8,0.49
6962,content_f01216059a6a,client_4ec9599fc2,71.486481,no_flag,monitor,52,down,335,5.3,3.85
6653,content_5fe46e04994d,client_4e07408562,71.152815,declining_with_demand,review,517715,down,104,4.2,0.14


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1) content_7f116ae1f6f5 — flagged for refresh (stale_visible_page). 954
impressions, 301 days stale, position 9. Would be wrong if this content was deliberately left unchanged (e.g. an evergreen reference page that doesn't need updates).


2) content_cf56e2e2e282 — flagged for refresh. Huge demand (61,678 impressions), 194 days stale, weak CTR (0.15%) at position ~20. Strong candidate; would be wrong if the low CTR is a snippet/SERP-feature issue unrelated to freshness.


3) content_72496874f806 — flagged for refresh. 821 impressions, 301 days stale, decent position (5.8). Would be wrong if the page's ranking is already stable and a refresh risks disrupting it.


4) content_7368877ea310 — flagged for refresh. Very high demand (59,472), weak CTR (0.13%) at position ~25. Would be wrong if the ranking (not the content) is the real bottleneck — refreshing content won't fix a position problem.


5) content_55a5b1c46474 — flagged no_flag/monitor, but scored 74.5 — nearly as high as top refresh picks despite only 35 impressions and 0% CTR. This looks like a scoring bug, not a genuine top candidate — see note below.


6) content_1bfaa38ff26c — flagged for refresh. 25,715 impressions, decent position (22.2%... wait, CTR 0.23%), 194 days stale. Would be wrong if this client has a policy against touching high-traffic pages without review.


7) content_02b0d6e30129 — flagged review (declining_with_demand). Modest demand (176), 0% CTR. Would be wrong if 176 impressions is too low to trust the decline as real vs. noise.


8) content_df1fa766cac2 — flagged review. 206 impressions, CTR 0.49%, declining. Would be wrong if this is a seasonal dip rather than a genuine decline (per the lane guide's consolidation/seasonality warning).


9) content_f01216059a6a — flagged no_flag/monitor, scored 71.5 with only 52 impressions but a notably high CTR (3.85%). Another case where a "no_flag" row is scoring suspiciously high — worth investigating.


10) content_5fe46e04994d — flagged review. Massive demand (517,715 impressions!), declining, but recently updated (104 days). Very strong
candidate — high stakes if this decline is real. Would be wrong if this reflects a deliberate traffic redirect to another page (consolidation).


11) content_f488400fca67 — flagged review. Small demand (155), 0% CTR, declining. Would be wrong if 155 impressions is too thin to trust.


12) content_0a91db491d14 — flagged refresh. Large demand (13,299), moderate CTR (0.49%), 193 days stale. Solid candidate.


13) content_7a888d3d99c8 — flagged no_flag/monitor, but scored 70.7 despite terrible position (67.6!) and 0% CTR — a page basically invisible in search.High staleness score is likely inflating this despite the page having no real visibility to lose. Another scoring-bug candidate.


14) content_2c2606c5d176 — flagged review. Very high demand (347,399), declining, recently updated. High-stakes candidate.


15) content_e2b702f4f92b — flagged no_flag/monitor, scored 69.9 with only 30 impressions. Same bug pattern as #5, #9, #13.


16) content_cb112fce36be — flagged review. Huge demand (309,910), declining. Strong candidate.


17) content_9532f197bbc8 — flagged review. Huge demand (309,192), excellent position (2.0!) but declining with weak-ish CTR (0.87%). Interesting — declining despite a great position; worth investigating why.


18) content_5feee3994adb — flagged refresh. Solid demand (7,812), very poor CTR (0.01%!), position 39. Would be wrong if this position is simply too far down for any refresh to matter.


19) content_c2d929d83eaa — flagged refresh. Solid demand (7,558), CTR 0.20%, position 17.9. Reasonable candidate.


20) content_07ce98c6085a — flagged no_flag/monitor, scored 69.4 with only 85 impressions, 0% CTR, high staleness (304 days). Same pattern again.




## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rows 5, 9, 13, 15, and 20 are all labeled no_flag/monitor, yet score in the same 69-75 range as genuinely flagged pages.

Several with tiny impression counts (30-95) or a terrible position (67.6). This reveals a scoring formula bug, not five bad content picks: staleness_score (normalized by days_since_last_update) is being weighted heavily (0.4) regardless of whether a page actually earned a reason code.

A page with almost no traffic can still score high purely by being old, even though it has nothing worth reviewing. Fix: either zero out the score for no_flag rows, or make the score conditional on at least one rule condition being true, rather than a pure weighted-average of raw signals.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.